#

In [ ]:
import math
import os
import re
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from google.colab import drive
drive.mount('/content/drive')


# Parameters

In [ ]:
DATA_PATH = "./drive/MyDrive/shakespeare.txt"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEQ_LEN = 64
BATCH_SIZE = 64
EPOCHS = 10
LR = 3e-4

MIN_FREQ = 2
MAX_VOCAB = 30000

EMB_DIM = 256
HIDDEN_DIM = 512
LAYERS = 2
HEADS = 8
DROPOUT = 0.2

SAVE_PATH = "lstm_attn_lm.pt"
print("Device:", DEVICE)


# Model definition

In [ ]:
class LSTMAttnLM(nn.Module):
    def __init__(
        self,
        vocab_size,
        emb_dim=256,
        hidden_dim=512,
        num_layers=2,
        attn_heads=8,
        dropout=0.2,
        pad_id=0,
    ):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attn = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=attn_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.fuse = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.lm_head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        # x: [B - batch_size, T - seq_len]
        B, T = x.shape
        e = self.emb(x)      # [B, T, E]
        h, _ = self.lstm(e)  # [B, T, H]

        # causal mask: mask out j>i
        attn_mask = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)  # [B, T, H]

        z = self.fuse(torch.cat([h, a], dim=-1))  # [B, T, H]
        logits = self.lm_head(z)                  # [B, T, V]
        return logits


# Help funcs

In [ ]:

TOKEN_RE = re.compile(r"\w+|[^\w\s]", re.UNICODE)

SPECIALS = ["<pad>", "<unk>", "<bos>", "<eos>"]
PAD, UNK, BOS, EOS = SPECIALS

def tokenize(text: str):
    return TOKEN_RE.findall(text.lower())

def build_vocab(tokens, min_freq=2, max_size=None):
    cnt = Counter(tokens)
    items = sorted(cnt.items(), key=lambda x: (-x[1], x[0]))

    stoi = {s: i for i, s in enumerate(SPECIALS)}
    itos = list(SPECIALS)

    for tok, f in items:
        if f < min_freq:
            continue
        if tok in stoi:
            continue
        if max_size is not None and len(itos) >= max_size:
            break
        stoi[tok] = len(itos)
        itos.append(tok)
    return stoi, itos

def numericalize(tokens, stoi):
    unk_id = stoi[UNK]
    return [stoi.get(t, unk_id) for t in tokens]


# Loading the dataset

In [ ]:
class TextWindowDataset(Dataset):
    def __init__(self, ids, seq_len):
        self.ids = torch.tensor(ids, dtype=torch.long)
        self.seq_len = seq_len
        self.n = len(self.ids) - seq_len
        self.n = max(self.n, 0)

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        chunk = self.ids[i : i + self.seq_len + 1]
        x = chunk[:-1]  # [T]
        y = chunk[1:]   # [T]
        return x, y


In [6]:
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError(f"Nie widzę pliku: {DATA_PATH}")

raw = open(DATA_PATH, "r", encoding="utf-8", errors="ignore").read()
tokens = tokenize(raw) + [EOS]

stoi, itos = build_vocab(tokens, min_freq=MIN_FREQ, max_size=MAX_VOCAB)
ids = numericalize(tokens, stoi)

PAD_ID = stoi[PAD]
VOCAB_SIZE = len(itos)

print("Num tokens:", len(tokens))
print("Vocab size:", VOCAB_SIZE)
print("Example tokens:", tokens[:30])


Num tokens: 1172762
Vocab size: 14907
Example tokens: ['from', 'fairest', 'creatures', 'we', 'desire', 'increase', ',', 'that', 'thereby', 'beauty', "'", 's', 'rose', 'might', 'never', 'die', ',', 'but', 'as', 'the', 'riper', 'should', 'by', 'time', 'decease', ',', 'his', 'tender', 'heir', 'might']


In [7]:
ds = TextWindowDataset(ids, SEQ_LEN)
len(ds)

1172698

In [8]:
n = len(ds)
n_train = int(0.95 * n)
n_val = n - n_train

train_ds, val_ds = torch.utils.data.random_split(ds, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print("Train windows:", len(train_ds))
print("Val windows:", len(val_ds))


Train windows: 1114063
Val windows: 58635


# Training and Evaluation

In [9]:
model = LSTMAttnLM(
    vocab_size=VOCAB_SIZE,
    emb_dim=EMB_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=LAYERS,
    attn_heads=HEADS,
    dropout=DROPOUT,
    pad_id=PAD_ID,
).to(DEVICE)

sum(p.numel() for p in model.parameters())

16717115

In [10]:
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)

def run_epoch(model, loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    total_tokens = 0

    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        with torch.set_grad_enabled(train):
            logits = model(x)  # [B, T, V]
            loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), y.reshape(-1))

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

        non_pad = (y != PAD_ID).sum().item()
        total_loss += loss.item() * max(non_pad, 1)
        total_tokens += max(non_pad, 1)

    avg_loss = total_loss / max(total_tokens, 1)
    ppl = math.exp(min(avg_loss, 20.0))
    return avg_loss, ppl


In [11]:
@torch.no_grad()
def generate(model,prompt,max_new_tokens=80,temperature=1.0, top_k=0):
    model.eval()

    prompt_ids = numericalize(tokenize(prompt), stoi)
    ids_local = [stoi[BOS]] + prompt_ids

    for _ in range(max_new_tokens):
        ctx = ids_local[-SEQ_LEN:]
        x = torch.tensor(ctx, dtype=torch.long, device=DEVICE).unsqueeze(0)  # [1, T]
        logits = model(x)[:, -1, :]  # [1, V]
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            v, ix = torch.topk(logits, k=min(top_k, logits.shape[-1]), dim=-1)
            masked = torch.full_like(logits, float("-inf"))
            masked.scatter_(1, ix, v)
            logits = masked

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1).item()
        ids_local.append(next_id)

    out = [itos[i] for i in ids_local if itos[i] != BOS]

    # prosta reguła spacji przed interpunkcją
    text = ""
    for t in out:
        if not text:
            text = t
        elif re.fullmatch(r"[.,;:!?)\]]", t):
            text += t
        elif re.fullmatch(r"[(\[]", t):
            text += " " + t
        else:
            text += " " + t
    return text

print(generate(model, "from fairest creatures", max_new_tokens=30, temperature=1.0, top_k=50))


from fairest creatures drums pitch quote fertile tish recourse cureless brows weapon epithet cognizance warwick break reveng dragging threaden recourse hazarded dragging dimm warwick scratch wagging dragging brows tish unity inferior unity otter


In [ ]:
# Cell 10: Training
best_val = float("inf")

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_ppl = run_epoch(model, train_loader, train=True)
    va_loss, va_ppl = run_epoch(model, val_loader, train=False)

    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} ppl {tr_ppl:.2f} | val loss {va_loss:.4f} ppl {va_ppl:.2f}")

    if va_loss < best_val:
        best_val = va_loss
        torch.save(
            {
                "model_state": model.state_dict(),
                "stoi": stoi,
                "itos": itos,
                "config": {
                    "SEQ_LEN": SEQ_LEN,
                    "EMB_DIM": EMB_DIM,
                    "HIDDEN_DIM": HIDDEN_DIM,
                    "LAYERS": LAYERS,
                    "HEADS": HEADS,
                    "DROPOUT": DROPOUT,
                },
            },
            SAVE_PATH,
        )
        print("Saved:", SAVE_PATH)

    print(generate(model, "from fairest creatures", max_new_tokens=60, temperature=0.9, top_k=50))
    print()


# Testing the model

In [ ]:
# Cell 11: Load best checkpoint (opcjonalnie)
ckpt = torch.load(SAVE_PATH, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
stoi = ckpt["stoi"]
itos = ckpt["itos"]
print("Loaded:", SAVE_PATH)


In [ ]:
# Cell 12: Final sampling
print(generate(model, "when forty winters", max_new_tokens=120, temperature=0.9, top_k=50))
